In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import os

print("🎬 Initializing Real-Data XAI Animation Studio...")

# ==========================================
# 1. KINECT V2 ANATOMY DEFINITION
# ==========================================
kinect_bones = [
    (0, 1), (1, 20), (20, 2), (2, 3),                     # Spine to Head
    (20, 4), (4, 5), (5, 6), (6, 7), (7, 21), (7, 22),    # Left Arm & Hand
    (20, 8), (8, 9), (9, 10), (10, 11), (11, 23), (11, 24), # Right Arm & Hand
    (0, 12), (12, 13), (13, 14), (14, 15),                # Left Leg
    (0, 16), (16, 17), (17, 18), (18, 19)                 # Right Leg
]

# ==========================================
# 2. THE DATA PARSER
# ==========================================
def parse_single_skeleton(file_path):
    with open(file_path, 'r') as f:
        datas = f.readlines()
    if not datas: return None

    nframe = int(datas[0].strip())
    skeleton_tensor = np.zeros((nframe, 2, 25, 3), dtype=np.float32)
    
    cursor = 0
    for frame in range(nframe):
        cursor += 1
        bodycount = int(datas[cursor].strip())
        if bodycount == 0: continue
            
        for body in range(bodycount):
            cursor += 2 
            njoints_in_file = int(datas[cursor].strip())
            for joint in range(njoints_in_file):
                cursor += 1 
                if body < 2: 
                    jointinfo = datas[cursor].strip().split()
                    skeleton_tensor[frame, body, joint, :] = [float(jointinfo[0]), float(jointinfo[1]), float(jointinfo[2])]
    return skeleton_tensor

# ==========================================
# 3. LOAD REAL DATA & SIMULATE XAI
# ==========================================
# UPDATE THIS PATH TO A REAL FILE ON YOUR COMPUTER!
file_path = '../data/raw_skeletons/nturgb+d_skeletons_s001_to_s017/S001C001P001R001A010.skeleton' 

try:
    raw_data = parse_single_skeleton(file_path)
    # Extract Person 1 only (Shape: Frames, Joints, Coordinates)
    person_sequence = raw_data[:, 0, :, :] 
    num_frames = person_sequence.shape[0]
    print(f"✅ Loaded real data: {num_frames} frames.")
except Exception as e:
    print(f"❌ Error loading file: Make sure the path is correct. ({e})")
    exit()

# Create the fake XAI Heatmap Sequence (0.0 to 1.0)
# We will simulate the AI figuring out the action over time
xai_heatmaps = np.random.rand(num_frames, 25) * 0.2 
for t in range(num_frames):
    intensity = (t / num_frames) 
    xai_heatmaps[t, 11] = 0.5 + (intensity * 0.5) # Right Hand gets HOT
    xai_heatmaps[t, 10] = 0.4 + (intensity * 0.5) # Right Wrist gets warm
    xai_heatmaps[t, 3] = 0.6                      # Head stays warm

# ==========================================
# 4. BUILD THE ANIMATION ENGINE
# ==========================================
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
ax.set_title("Temporal Brain XAI: Joint Attention Map", fontsize=16, fontweight='bold')
ax.set_axis_off() 

# Auto-scale the camera based on the real person's movements
x_min, x_max = np.min(person_sequence[:, :, 0]), np.max(person_sequence[:, :, 0])
y_min, y_max = np.min(person_sequence[:, :, 1]), np.max(person_sequence[:, :, 1])
z_min, z_max = np.min(person_sequence[:, :, 2]), np.max(person_sequence[:, :, 2])

# Pad the limits slightly so the skeleton doesn't hit the edge of the screen
ax.set_xlim([x_min - 0.2, x_max + 0.2])
# NOTE: Kinect Y is height, Z is depth. We swap them in Matplotlib so the person stands upright!
ax.set_ylim([z_min - 0.2, z_max + 0.2]) 
ax.set_zlim([y_min - 0.2, y_max + 0.2]) 

# Adjust camera angle for best viewing
ax.view_init(elev=0, azim=-90) 

bone_lines = [ax.plot([], [], [], color='black', linewidth=3, zorder=1)[0] for _ in kinect_bones]
joint_scatter = ax.scatter([], [], [], s=120, zorder=2, edgecolors='white', linewidth=1.5)

cmap = cm.get_cmap('coolwarm') 

def update(frame_idx):
    current_skeleton = person_sequence[frame_idx]
    current_heat = xai_heatmaps[frame_idx]
    
    # 1. Update Bones (Swapping Y and Z for Matplotlib orientation)
    for i, (j1, j2) in enumerate(kinect_bones):
        x_vals = [current_skeleton[j1, 0], current_skeleton[j2, 0]]
        y_vals = [current_skeleton[j1, 2], current_skeleton[j2, 2]] # Depth becomes Y
        z_vals = [current_skeleton[j1, 1], current_skeleton[j2, 1]] # Height becomes Z
        bone_lines[i].set_data(x_vals, y_vals)
        bone_lines[i].set_3d_properties(z_vals)
        
    # 2. Update Joints
    joint_scatter._offsets3d = (current_skeleton[:, 0], current_skeleton[:, 2], current_skeleton[:, 1])
    
    # 3. Apply XAI Colors
    colors = cmap(current_heat)
    joint_scatter.set_color(colors)
    joint_scatter.set_edgecolor('black') 
    
    return bone_lines + [joint_scatter]

print("⏳ Rendering GIF with real data... (This may take 30-60 seconds)")
ani = FuncAnimation(fig, update, frames=num_frames, interval=33, blit=False) # 33ms interval = ~30fps

gif_filename = 'real_xai_simulation.gif'
ani.save(gif_filename, writer='pillow', fps=30)
plt.close()

print(f"✅ Success! Animation saved locally as: {gif_filename}")

🎬 Initializing Real-Data XAI Animation Studio...
✅ Loaded real data: 69 frames.
⏳ Rendering GIF with real data... (This may take 30-60 seconds)


C:\Users\ABEL\AppData\Local\Temp\ipykernel_5968\1360113331.py:97: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('coolwarm')


✅ Success! Animation saved locally as: real_xai_simulation.gif
